# Set up

In [1]:
import os, sys, json, datetime, re  # Provides OS-dependent functionality, system-specific parameters, JSON handling, and date/time manipulation
import pandas as pd             # Provides data structures and data analysis tools
import numpy as np              # Supports large, multi-dimensional arrays and matrices
import requests
import time
from tqdm import tqdm
import glob as glob

import csv
from pathlib import Path

#thi data contants
from cprl_functions.defined_functions import *
from cprl_functions.state_capture import *
from cprl_functions.text_printing import bordered
from cprl_functions.data_packet_defs import *

###################
import data_sources.data_collection.pull_data as data_pull
from graphs.viz_graphs_template import *


In [2]:
# path = r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data Packets\K-12\graphs'
# print_directory_tree(path)

## get the graphs

In [11]:
#Mapping and getting Graphs funcitons
import os
import pandas as pd
from pathlib import Path

# Define the mapping between graph files and column tags
GRAPH_MAPPINGS = {
    '1.1.png': '@num_trad',
    '1.2.png': '@k-12_race',
    '1.3.png': '@k-12_frl',
    '1.4.png': '@k-12_enrollment',
    '2.1.png': '@state-achievement_grade4',
    '2.2.png': '@state-achievement_grade8',
    '3.1.png': '@NAEP_grade4_reading',
    '3.2.png': '@NAEP_grade8_reading',
    '3.3.png': '@NAEP_grade4_math',
    '3.4.png': '@NAEP_grade8_math',
    '4.1.png': '@NAEP_Region1',
    '4.2.png': '@NAEP_Region2',
    '4.3.png': '@NAEP_Region3',
    '4.4.png': '@NAEP_Region4',
    '4.5.png': '@NAEP_state_proficiency_reading',
    '4.6.png': '@NAEP_state_proficiency_math',
    '5.1.png': '@NAEP_RE_grade4reading',
    '5.2.png': '@NAEP_RE_grade4math',
    '5.3.png': '@NAEP_FRL_Reading',
    '5.4.png': '@NAEP_FRL_Math',
    '5.5.png': '@NAEP_ELL_Reading',
    '5.6.png': '@NAEP_ELL_Math',
    '5.7.png': '@NAEP_SWD_Reading',
    '5.8.png': '@NAEP_SWD_Math',
    '6.1.png': '@chronic_absenteeism_race',
    '6.2.png': '@chronic_absenteeism_other',
    '6.3.png': '@suspension',
    '7.1.png': '@HS_grad',
    '7.2.png': '@HS_grad_race',
    '7.3.png': '@HS_grad_other',
    '8.1.png': '@dropouts',
    '8.2.png': '@ccr_race',
    '8.3.png': '@AP1',
    '8.4.png': '@AP2',
    '8.5.png': '@AP3',
    '9.1.png': '@college_entrance_exam'
}

def scan_state_directories(main_graphs_dir):
    """
    Scans all state directories and maps available graphs to column tags.
    
    Args:
        main_graphs_dir: Path to the main directory containing state subdirectories
    
    Returns:
        Dictionary with state codes as keys and their graph mappings as values
    """
    main_path = Path(main_graphs_dir)
    state_mappings = {}
    
    if not main_path.exists():
        print(f"Error: Directory '{main_graphs_dir}' does not exist")
        return state_mappings
    
    # Iterate through all subdirectories (state folders)
    for state_dir in sorted(main_path.iterdir()):
        if state_dir.name not in state_abbreviations_priority:
            continue
        if state_dir.is_dir():
            state_code = state_dir.name
            state_mappings[state_code] = {}
            
            # Check which graphs exist for this state
            for graph_file, column_tag in GRAPH_MAPPINGS.items():
                graph_path = state_dir / graph_file
                if graph_path.exists():
                    # Store relative path from main directory
                    state_mappings[state_code][column_tag] = str(graph_path.resolve())
    
    return state_mappings

def generate_merge_dataframe(state_mappings):
    """
    Generates a pandas DataFrame with all column tags and their corresponding graph paths.
    
    Args:
        state_mappings: Dictionary from scan_state_directories
    
    Returns:
        pandas DataFrame with all columns
    """
    # All column tags from your header
    all_columns = [
        '@num_trad', '@k-12_race', '@k-12_frl','@k-12_enrollment',
        '@state-achievement_grade4', '@state-achievement_grade8',
        '@NAEP_grade4_reading', '@NAEP_grade8_reading',
        '@NAEP_grade4_math', '@NAEP_grade8_math',
        '@NAEP_Region1', '@NAEP_Region2', '@NAEP_Region3', '@NAEP_Region4',
        '@NAEP_state_proficiency_reading', '@NAEP_state_proficiency_math',
        '@NAEP_RE_grade4reading', '@NAEP_RE_grade4math',
        '@NAEP_FRL_Reading', '@NAEP_FRL_Math',
        '@NAEP_ELL_Reading', '@NAEP_ELL_Math',
        '@NAEP_SWD_Reading', '@NAEP_SWD_Math',
        '@chronic_absenteeism_race', '@chronic_absenteeism_other', '@suspension',
        '@HS_grad', '@HS_grad_race', '@HS_grad_other',
        '@dropouts', '@ccr_race',
        '@AP1', '@AP2', '@AP3',
        '@college_entrance_exam'
    ]
    # Build list of rows
    rows = []
    for state_code in sorted(state_mappings.keys()):
        row = {'state_abrv': state_code}
        
        # Fill in graph paths for this state
        for column_tag, graph_path in state_mappings[state_code].items():
            row[column_tag] = graph_path
        
        rows.append(row)
        
    # Create DataFrame
    df = pd.DataFrame(rows, columns=['state_abrv'] + all_columns)
    
    return df

def print_summary(state_mappings):
    """Print a summary of what was found."""
    print(f"\nFound {len(state_mappings)} state directories")
    print("\nSummary by state:")
    for state, mappings in sorted(state_mappings.items()):
        print(f"  {state}: {len(mappings)} graphs found")
    
    # Check for missing graphs
    all_tags = set(GRAPH_MAPPINGS.values())
    print(f"\nTotal unique graph types: {len(all_tags)}")


In [12]:
#getting the graph file paths
MAIN_GRAPHS_DIR = r"C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs"

print("Scanning state directories...")
state_mappings = scan_state_directories(MAIN_GRAPHS_DIR)
print(state_mappings)


Scanning state directories...
{'AZ': {'@num_trad': 'C:\\Users\\clutz\\THE HUNT INSTITUTE\\The Hunt Institute Team Site - Documents\\Policy Team\\Data\\Data Packets\\K-12\\graphs\\AZ\\1.1.png', '@k-12_race': 'C:\\Users\\clutz\\THE HUNT INSTITUTE\\The Hunt Institute Team Site - Documents\\Policy Team\\Data\\Data Packets\\K-12\\graphs\\AZ\\1.2.png', '@k-12_frl': 'C:\\Users\\clutz\\THE HUNT INSTITUTE\\The Hunt Institute Team Site - Documents\\Policy Team\\Data\\Data Packets\\K-12\\graphs\\AZ\\1.3.png', '@k-12_enrollment': 'C:\\Users\\clutz\\THE HUNT INSTITUTE\\The Hunt Institute Team Site - Documents\\Policy Team\\Data\\Data Packets\\K-12\\graphs\\AZ\\1.4.png', '@state-achievement_grade4': 'C:\\Users\\clutz\\THE HUNT INSTITUTE\\The Hunt Institute Team Site - Documents\\Policy Team\\Data\\Data Packets\\K-12\\graphs\\AZ\\2.1.png', '@state-achievement_grade8': 'C:\\Users\\clutz\\THE HUNT INSTITUTE\\The Hunt Institute Team Site - Documents\\Policy Team\\Data\\Data Packets\\K-12\\graphs\\AZ\\2.

In [13]:

if state_mappings:
    print_summary(state_mappings)

    # Generate DataFrame
    graphs_df = generate_merge_dataframe(state_mappings)


else:
    print("No state directories found or directory doesn't exist.")
    print(f"Make sure '{MAIN_GRAPHS_DIR}' exists and contains state subdirectories.")


Found 34 state directories

Summary by state:
  AZ: 36 graphs found
  CT: 36 graphs found
  DC: 36 graphs found
  DE: 36 graphs found
  GA: 36 graphs found
  IA: 36 graphs found
  ID: 35 graphs found
  IL: 36 graphs found
  IN: 36 graphs found
  KS: 36 graphs found
  KY: 35 graphs found
  MA: 36 graphs found
  MD: 36 graphs found
  MI: 35 graphs found
  MN: 35 graphs found
  MO: 36 graphs found
  NC: 36 graphs found
  ND: 36 graphs found
  NE: 35 graphs found
  NJ: 36 graphs found
  NM: 36 graphs found
  NY: 35 graphs found
  OH: 36 graphs found
  OK: 36 graphs found
  OR: 36 graphs found
  RI: 35 graphs found
  SC: 36 graphs found
  TN: 35 graphs found
  TX: 35 graphs found
  VA: 36 graphs found
  VT: 36 graphs found
  WA: 35 graphs found
  WV: 36 graphs found
  WY: 36 graphs found

Total unique graph types: 36


In [14]:
print(graphs_df.to_string(max_colwidth=20))

   state_abrv            @num_trad           @k-12_race            @k-12_frl     @k-12_enrollment @state-achievement_grade4 @state-achievement_grade8 @NAEP_grade4_reading @NAEP_grade8_reading    @NAEP_grade4_math    @NAEP_grade8_math        @NAEP_Region1        @NAEP_Region2        @NAEP_Region3        @NAEP_Region4 @NAEP_state_proficiency_reading @NAEP_state_proficiency_math @NAEP_RE_grade4reading  @NAEP_RE_grade4math    @NAEP_FRL_Reading       @NAEP_FRL_Math    @NAEP_ELL_Reading       @NAEP_ELL_Math    @NAEP_SWD_Reading       @NAEP_SWD_Math @chronic_absenteeism_race @chronic_absenteeism_other          @suspension             @HS_grad        @HS_grad_race       @HS_grad_other            @dropouts            @ccr_race                 @AP1                 @AP2                 @AP3 @college_entrance_exam
0          AZ  C:\Users\clutz\T...  C:\Users\clutz\T...  C:\Users\clutz\T...  C:\Users\clutz\T...  C:\Users\clutz\T...       C:\Users\clutz\T...       C:\Users\clutz\T...  C:\Users\clutz

In [ ]:
import shutil
from pathlib import Path

source_dir = r"C:\Users\clutz\Downloads\1.4"

destination_dir = r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs'

files = glob.glob(os.path.join(source_dir, '*'))
for file in files:
    filename = file.split('\\')[-1]
    state = filename.split('_')[-1].replace('.jpg','')
    # print(filename)
    state_abrv = state_ref_lower.get(state.lower())
    destination_filepath = os.path.join(destination_dir,state_abrv,'1.4.png')
    print(destination_filepath)
    # shutil.move(file, destination_filepath)
    
    # print(state_abrv)


C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs\AZ\1.4.png
C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs\CT\1.4.png
C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs\DE\1.4.png
C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs\DC\1.4.png
C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs\GA\1.4.png
C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs\IL\1.4.png
C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\graphs\IN\1.4.png
C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets

## Convert File Paths

In [17]:
def convert_path(old_path, old_root, new_root):
    # Remove old root
    if isinstance(old_path, float):
        return None
    if old_path.startswith(old_root):
        relative_path = old_path[len(old_root):].lstrip(r"\/")  # remove leading slash/backslash
        # Replace backslashes with colons
        relative_path_colon = relative_path.replace("\\", ":")
        # Join with new root
        new_path = f"{new_root}:{relative_path_colon}"
        return new_path
    else:
        return old_path  # unchanged if it doesn't match
# def convert_path(old_path, old_root, new_root):
#     # Remove old root
#     if old_path.startswith(old_root):
#         relative_path = old_path[len(old_root):].lstrip(r"\/")
#         # Use forward slashes for Mac paths (POSIX style)
#         relative_path_slash = relative_path.replace("\\", "/")
#         # Join with new root using forward slash
#         new_path = f"{new_root}/{relative_path_slash}"
#         return new_path
#     else:
#         return old_path    




In [18]:
import os

# Original root and new root
old_root = r"C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents"
new_root = r"Macintosh HD:Users:bavery:Library:CloudStorage:OneDrive-SharedLibraries-THEHUNTINSTITUTE-ViaTA:The Hunt Institute Team Site - Documents"
# new_root = "Users/bavery/Library/CloudStorage/OneDrive-TheHuntInstitute/Shared Documents"
df_converted = graphs_df.map(lambda x: convert_path(x, old_root, new_root))
# or for older pandas:
# df_converted = graphs_df.applymap(lambda x: convert_path(x, old_root, new_root))
# graphs_df['']
merge_graphs_dir = r'C:\Users\clutz\THE HUNT INSTITUTE\The Hunt Institute Team Site - Documents\Policy Team\Data\Data Packets\K-12\merge_files\graphs'
df_converted.to_csv(os.path.join(merge_graphs_dir,'graphs_merge_file.csv'), index=False)

print(df_converted.to_string())


   state_abrv                                                                                                                                                                                     @num_trad                                                                                                                                                                                    @k-12_race                                                                                                                                                                                     @k-12_frl                                                                                                                                                                              @k-12_enrollment                                                                                                                                                                     @state-achievement_grade4                                     